# 🧱 Data Forge — quick tour

Shows how to ingest mixed documents (PDF, DOCX, XLSX, images + OCR...) and build a training dataset with the correct chat template for **any** domain you care about.

The notebook is **not tied to `asset_integrity`** — set `DOMAIN_NAME` to whatever engagement you're working on.

Run from the repo root.

In [ ]:
# ── Environment bootstrap (Colab / Brev / local) ──
# Clones the repo into /content on Colab/Brev so `from app...`
# resolves. On a local checkout the repo already exists and the
# clone is skipped; we just chdir into it.
import os, sys, subprocess, pathlib

REPO_URL  = 'https://github.com/valonys/finetuningtraining.git'
REPO_NAME = 'finetuningtraining'
BRANCH    = 'develop'

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _find_repo_root() -> pathlib.Path | None:
    here = pathlib.Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'app' / '__init__.py').exists():
            return candidate
    return None

root = _find_repo_root()
if root is None:
    target = pathlib.Path('/content') if _in_colab() else pathlib.Path.cwd()
    dest = target / REPO_NAME
    if not dest.exists():
        print(f'📥 Cloning {REPO_URL} ({BRANCH}) -> {dest}')
        subprocess.check_call(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(dest)]
        )
    root = dest

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print(f'✅ Repo root: {root}')

if _in_colab():
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers>=4.44', 'trl>=0.11', 'peft>=0.12',
        'datasets>=2.20', 'accelerate>=0.33', 'pyyaml', 'pydantic>=2.0',
    ])
    print('✅ Dependencies installed')

In [ ]:

DOMAIN_NAME   = 'ai_llm'                     # ← change per engagement
BASE_MODEL    = 'Qwen/Qwen2.5-7B-Instruct'
SYSTEM_PROMPT = 'You are an expert in the chosen domain. Be precise and cite sources.'

from app.config_loader import create_domain_config, domain_config_exists, load_domain_config

if not domain_config_exists(DOMAIN_NAME):
    create_domain_config(name=DOMAIN_NAME, system_prompt=SYSTEM_PROMPT)
    print(f'✅ Created configs/domains/{DOMAIN_NAME}.yaml')

config = load_domain_config(DOMAIN_NAME)
SYSTEM_PROMPT = config['system_prompt']

In [ ]:
from app.data_forge import DataForge

forge = DataForge()
records = forge.ingest('./data/uploads/')
print(f'Ingested {len(records)} records')
for r in records[:3]:
    print(r.source_type, '-', r.source[:80], '-', len(r.text), 'chars')

In [ ]:
ds = forge.build_dataset(
    records,
    task='sft',
    base_model=BASE_MODEL,
    system_prompt=SYSTEM_PROMPT,
    synth_qa=True,
    target_size=200,
)
print(f'Rows: {len(ds)}')
print(ds[0]['text'][:800])
ds.save_to_disk(f'./data/processed/{DOMAIN_NAME}_sft')